In [57]:
# -*- coding: utf-8 -*-
"""
三方演化博弈完整实现 - AI企业、用户、政府监管
包含：符号计算、动态仿真、稳定性分析、二维时间序列可视化
（以 400 dpi 的 JPG 格式保存到当前目录）
"""
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# ================ 修复中文显示 ================
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

from scipy.integrate import odeint

# ================== 符号定义 ==================
x, y, z = sp.symbols('x y z')
Bg, Dg, F, alpha, R1, R2, C1, C2, C3, V1, V2, L1, L2 = sp.symbols(
    'Bg Dg F alpha R1 R2 C1 C2 C3 V1 V2 L1 L2'
)

# ================== 复制动态方程 ==================
def replicator_dynamics():
    dx = x * (1 - x) * (y*(R1 + L1) + z*F*(1 + alpha) - C1)
    dy = y * (1 - y) * (x*(V1 + V2) - V2 - C2 + L2)
    dz = z * (1 - z) * (x*y*R2 + Bg + Dg - C3)
    return [dx, dy, dz]

# ================== 雅可比矩阵 ==================
def compute_jacobian():
    eqs = replicator_dynamics()
    return sp.Matrix([
        [sp.simplify(sp.diff(eq, var)) for var in (x, y, z)]
        for eq in eqs
    ])

# ================== 稳定性分析 ==================
def analyze_stability(jacobian, params=None):
    equilibrium_points = [
        (0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1),
        (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)
    ]

    subs_dict = {}
    if params:
        subs_dict = {
            Bg: params['Bg'], Dg: params['Dg'], F: params['F'],
            alpha: params['alpha'], R1: params['R1'], R2: params['R2'],
            C1: params['C1'], C2: params['C2'], C3: params['C3'],
            V1: params['V1'], V2: params['V2'], L1: params['L1'],
            L2: params['L2']
        }

    results = {}
    for point in equilibrium_points:
        J = jacobian.subs({x: point[0], y: point[1], z: point[2]})
        if subs_dict:
            J = J.subs(subs_dict)

        try:
            J_np = np.array(J.tolist(), dtype=float)
            eigvals = np.linalg.eigvals(J_np)
            results[point] = {
                'type': 'numerical',
                'eigenvalues': eigvals,
                'stable': np.all(np.real(eigvals) < 0)
            }
        except Exception:
            eig_dict = J.eigenvals()
            eig_list = []
            for val, count in eig_dict.items():
                eig_list.extend([val] * count)
            results[point] = {
                'type': 'symbolic',
                'eigenvalues': eig_list,
                'conditions': [sp.re(v) < 0 for v in eig_list]
            }

    return results

# ================== 数值仿真 ==================
def dynamic_simulation(ic, t, params):
    subs_dict = {
        Bg: params['Bg'], Dg: params['Dg'], F: params['F'],
        alpha: params['alpha'], R1: params['R1'], R2: params['R2'],
        C1: params['C1'], C2: params['C2'], C3: params['C3'],
        V1: params['V1'], V2: params['V2'], L1: params['L1'],
        L2: params['L2']
    }

    dx, dy, dz = replicator_dynamics()
    dx_f = sp.lambdify((x, y, z), dx.subs(subs_dict), 'numpy')
    dy_f = sp.lambdify((x, y, z), dy.subs(subs_dict), 'numpy')
    dz_f = sp.lambdify((x, y, z), dz.subs(subs_dict), 'numpy')

    def system(state, t):
        return [dx_f(*state), dy_f(*state), dz_f(*state)]

    return odeint(system, ic, t)

# ================== 可视化（仅新增保存，不改配色） ==================
def plot_time_series(t, data, title="0", filename="time_series.jpg"):
    fig, ax = plt.subplots(figsize=(12, 8))

    # ======= 原始配色，严格保持不变 =======
    ax.plot(t, data[:, 0], lw=2, color='#6AB4DC',
            label='AI Enterprise', marker='*', markersize=12)
    ax.plot(t, data[:, 1], lw=2, color='#42B8B1',
            label='User', marker='s', markersize=12)
    ax.plot(t, data[:, 2], lw=2, color='#E78489',
            label='Government Regulator', marker='^', markersize=12)

    ax.set_xlim(t[0], t[-1])
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('$t$', fontsize=32)
    ax.set_ylabel('Ratio', fontsize=32)
    
    # ======= 设置坐标轴刻度标签字体大小 ========
    ax.tick_params(axis='both', which='major', labelsize=32)
    ax.tick_params(axis='both', which='minor', labelsize=28)

    ax.grid(True, alpha=0.1, linestyle='--')
    ax.set_title(title, fontsize=32, pad=20)
    ax.legend(
        fontsize=32,
        loc='center right',      # 右侧中间
        frameon=True             # 与参考图一致，保留图例边框
    )

    plt.tight_layout()

    # ======= 仅新增：400 dpi JPG 保存 =======
    plt.savefig(
        filename,
        dpi=400,
        format='jpg',
        bbox_inches='tight'
    )
    plt.close()

# ================== 输出 ==================
def print_jacobian_matrix(jacobian):
    print("\n" + "=" * 80)
    print("雅可比矩阵（符号形式）")
    print("=" * 80)
    sp.pprint(jacobian)

def print_stability(results):
    print("\n" + "=" * 60)
    print("均衡点稳定性分析报告")
    print("=" * 60)
    for point, info in results.items():
        print(f"\n均衡点 {point}:")
        if info['type'] == 'numerical':
            print("  类型: 数值分析")
            print(f"  特征值: {np.round(info['eigenvalues'], 4)}")
            print(f"  稳定性: {'稳定' if info['stable'] else '不稳定'}")
        else:
            print("  类型: 符号分析")
            for cond in info['conditions']:
                print(f"  → {sp.pretty(cond)}")

# ================== 主程序 ==================
if __name__ == "__main__":
    params = {
        'Bg': 2,      # 政府监管收益
        'Dg': 1,      # 政府不监管损失
        'F': 3,       # 合规奖励
        'alpha': 0.5, # 惩罚强度系数
        'R1': 3,      # AI企业真实信息被采纳的收益
        'R2': 2,      # 政府因真实信息获得的社会效益
        'C1': 10,     # AI企业生成真实信息的成本
        'C2': 4,      # 用户采纳AI信息的成本
        'C3': 8,      # 政府监管成本
        'V1': 2,      # 准确信息对用户的正价值
        'V2': 2,      # 虚假信息对用户的负影响
        'L1': 3,      # AI企业生成虚假信息的声誉损失
        'L2': 8       # 用户未采纳AI信息的损失
    }


    jac = compute_jacobian()
    print_jacobian_matrix(jac)

    stability_results = analyze_stability(jac, params=params)
    print_stability(stability_results)

    t = np.linspace(0, 10, 100)
    trajectory = dynamic_simulation(
        ic=(0.5, 0.5, 0.5),
        t=t,
        params=params
    )

    plot_time_series(
        t,
        trajectory,
        title="",
        filename="(0,1,0).jpg"
    )


雅可比矩阵（符号形式）
⎡(1 - 2⋅x)⋅(-C₁ + F⋅z⋅(α + 1) + y⋅(L₁ + R₁))           -x⋅(L₁ + R₁)⋅(x - 1)   
⎢                                                                             
⎢           -y⋅(V₁ + V₂)⋅(y - 1)              (2⋅y - 1)⋅(C₂ - L₂ + V₂ - x⋅(V₁ 
⎢                                                                             
⎣              R₂⋅y⋅z⋅(1 - z)                             R₂⋅x⋅z⋅(1 - z)      

              -F⋅x⋅(α + 1)⋅(x - 1)       ⎤
                                         ⎥
+ V₂))                  0                ⎥
                                         ⎥
        (1 - 2⋅z)⋅(Bg - C₃ + Dg + R₂⋅x⋅y)⎦

均衡点稳定性分析报告

均衡点 (0, 0, 0):
  类型: 数值分析
  特征值: [-10.   2.  -5.]
  稳定性: 不稳定

均衡点 (0, 0, 1):
  类型: 数值分析
  特征值: [-5.5  2.   5. ]
  稳定性: 不稳定

均衡点 (0, 1, 0):
  类型: 数值分析
  特征值: [-4. -2. -5.]
  稳定性: 稳定

均衡点 (0, 1, 1):
  类型: 数值分析
  特征值: [ 0.5 -2.   5. ]
  稳定性: 不稳定

均衡点 (1, 0, 0):
  类型: 数值分析
  特征值: [10.  6. -5.]
  稳定性: 不稳定

均衡点 (1, 0, 1):
  类型: 数值分析
  特征值: [5.5 6.  5. ]
  稳定性: 不稳定

均